# ZeroJudge APCS Practice Solutions

這份 notebook 對應 `ZeroJudge_APCS_questions.ipynb`，包含解題想法與 Python 參考解法。

## f163. 貨物分配

### 解題想法

先記錄每個分裝站的左右子節點。儲貨站一開始已有重量，所以先用 DFS 從下往上算出每個節點的子樹總重量。

每放入一個貨物：

1. 從根節點 `1` 開始。
2. 比較左子樹與右子樹的目前總重量。
3. 往較輕的一邊走；一樣重則往左。
4. 到達儲貨站後，輸出該儲貨站編號。
5. 把貨物重量加到儲貨站以及剛剛經過的所有分裝站。

時間複雜度約為所有貨物實際走過路徑長度的總和。

In [ ]:
import sys

def solve(data: str) -> str:
    nums = list(map(int, data.split()))
    idx = 0
    n = nums[idx]
    m = nums[idx + 1]
    idx += 2

    # node n ~ 2n-1 are storage stations.
    weight = [0] * (2 * n)
    for node in range(n, 2 * n):
        weight[node] = nums[idx]
        idx += 1

    cargos = nums[idx:idx + m]
    idx += m

    left = [0] * (2 * n)
    right = [0] * (2 * n)

    for _ in range(n - 1):
        a, b, c = nums[idx:idx + 3]
        idx += 3
        left[a] = b
        right[a] = c

    # Compute subtree weights bottom-up with DFS from root.
    # A recursive DFS may hit recursion depth, so use an explicit stack.
    stack = [(1, False)]
    while stack:
        node, visited = stack.pop()
        if node >= n:
            continue
        if not visited:
            stack.append((node, True))
            stack.append((left[node], False))
            stack.append((right[node], False))
        else:
            weight[node] = weight[left[node]] + weight[right[node]]

    outputs = []

    for cargo in cargos:
        node = 1
        path = []

        # Walk until reaching a storage station.
        while node < n:
            path.append(node)
            l = left[node]
            r = right[node]
            if weight[l] <= weight[r]:
                node = l
            else:
                node = r

        outputs.append(str(node))

        # Add cargo weight to the reached station and every node on the route.
        weight[node] += cargo
        for p in path:
            weight[p] += cargo

    return " ".join(outputs)

if __name__ == "__main__":
    print(solve(sys.stdin.read()))


## f312. 人力分配

### 解題想法

因為 `n <= 100`，直接枚舉工廠一拿到幾個員工即可。

如果工廠一拿 `x1` 人，工廠二就拿 `n - x1` 人，計算總收益並更新最大值。

時間複雜度：`O(n)`。

In [ ]:
import sys

def solve(data: str) -> str:
    nums = list(map(int, data.split()))
    A1, B1, C1 = nums[0:3]
    A2, B2, C2 = nums[3:6]
    n = nums[6]

    best = None
    for x1 in range(n + 1):
        x2 = n - x1
        y1 = A1 * x1 * x1 + B1 * x1 + C1
        y2 = A2 * x2 * x2 + B2 * x2 + C2
        total = y1 + y2
        if best is None or total > best:
            best = total

    return str(best)

if __name__ == "__main__":
    print(solve(sys.stdin.read()))


## f605. 購買力

### 解題想法

對每個商品看三天價格：

- `max(prices) - min(prices) >= d` 才購買。
- 成本是三天平均。

時間複雜度：`O(n)`。

In [ ]:
import sys

def solve(data: str) -> str:
    nums = list(map(int, data.split()))
    idx = 0
    n = nums[idx]
    d = nums[idx + 1]
    idx += 2

    count = 0
    total_cost = 0

    for _ in range(n):
        prices = nums[idx:idx + 3]
        idx += 3

        if max(prices) - min(prices) >= d:
            count += 1
            total_cost += sum(prices) // 3

    return f"{count} {total_cost}"

if __name__ == "__main__":
    print(solve(sys.stdin.read()))


## g275. 七言對聯

### 解題想法

把每句七個數字存在 list。題目的第 2、4、6 個字，在 Python index 是 `1, 3, 5`。

依序檢查 A、B、C 三個規則，把違反的規則字母接起來。沒有違反就輸出 `None`。

In [ ]:
import sys

def violate_A(line):
    # positions 2, 4, 6 become indices 1, 3, 5
    return not (line[1] != line[3] and line[1] == line[5])

def solve(data: str) -> str:
    nums = list(map(int, data.split()))
    idx = 0
    n = nums[idx]
    idx += 1

    ans = []
    for _ in range(n):
        first = nums[idx:idx + 7]
        idx += 7
        second = nums[idx:idx + 7]
        idx += 7

        bad = ""

        if violate_A(first) or violate_A(second):
            bad += "A"

        # first line must end with 仄聲=1, second line must end with 平聲=0
        if not (first[6] == 1 and second[6] == 0):
            bad += "B"

        # positions 2, 4, 6 must be opposite between two lines
        if not (first[1] != second[1] and first[3] != second[3] and first[5] != second[5]):
            bad += "C"

        ans.append(bad if bad else "None")

    return "\n".join(ans)

if __name__ == "__main__":
    print(solve(sys.stdin.read()))


## g595. 修補圍籬

### 解題想法

掃過所有位置，只處理高度為 `0` 的壞掉圍籬。

- 左邊界：用右鄰居。
- 右邊界：用左鄰居。
- 中間：用左右鄰居較小值。

題目保證壞掉圍籬不相鄰，所以鄰居不會也是 `0`。

時間複雜度：`O(n)`。

In [ ]:
import sys

def solve(data: str) -> str:
    nums = list(map(int, data.split()))
    n = nums[0]
    h = nums[1:1 + n]

    total = 0
    for i in range(n):
        if h[i] != 0:
            continue

        if i == 0:
            total += h[i + 1]
        elif i == n - 1:
            total += h[i - 1]
        else:
            total += min(h[i - 1], h[i + 1])

    return str(total)

if __name__ == "__main__":
    print(solve(sys.stdin.read()))


## h027. 矩陣總和

### 解題想法

枚舉大矩陣 `B` 中每一個可能的左上角。對每個左上角，比較它對應的 `s × t` 子矩陣是否每個位置都和 `A` 差不超過 `r`。

若符合條件，就：

1. 計數加一。
2. 計算該子矩陣總和。
3. 更新與 `A` 總和差距的最小值。

這題限制不大，直接暴力檢查即可。

In [ ]:
import sys

def solve(data: str) -> str:
    nums = list(map(int, data.split()))
    idx = 0
    s, t, n, m, r = nums[idx:idx + 5]
    idx += 5

    A = []
    for _ in range(s):
        A.append(nums[idx:idx + t])
        idx += t

    B = []
    for _ in range(n):
        B.append(nums[idx:idx + m])
        idx += m

    sum_A = sum(map(sum, A))
    count = 0
    best_diff = None

    for top in range(n - s + 1):
        for left in range(m - t + 1):
            ok = True
            sub_sum = 0

            for i in range(s):
                if not ok:
                    break
                for j in range(t):
                    val = B[top + i][left + j]
                    if abs(A[i][j] - val) > r:
                        ok = False
                        break
                    sub_sum += val

            if ok:
                count += 1
                diff = abs(sub_sum - sum_A)
                if best_diff is None or diff < best_diff:
                    best_diff = diff

    return f"{count}\n{best_diff if best_diff is not None else -1}"

if __name__ == "__main__":
    print(solve(sys.stdin.read()))
